# 01_pretrain

『밑바닥부터 시작하는 딥러닝 ❻』 실습 코드 — 원본: `ch09/01_pretrain.py`

셀을 위에서부터 차례대로 실행하세요.

In [ ]:
import os, sys

# 노트북에는 __file__이 없으므로 pyproject.toml이 있는 폴더(저장소 루트)를 찾아 이동한다
_dir = os.path.abspath('.')
while not os.path.exists(os.path.join(_dir, 'pyproject.toml')) and _dir != os.path.dirname(_dir):
    _dir = os.path.dirname(_dir)
os.chdir(_dir)
if '.' not in sys.path:
    sys.path.append('.')
print('작업 폴더:', os.getcwd())

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.amp import autocast
from tqdm import tqdm
import matplotlib.pyplot as plt
import wandb
from webbot.model import GPT
from webbot.utils import get_device

In [ ]:
# --- DDP 초기화 ---
ddp = int(os.environ.get('RANK', -1)) != -1
if ddp:
    dist.init_process_group(backend='nccl')
    local_rank = int(os.environ['LOCAL_RANK'])
    device = torch.device(f'cuda:{local_rank}')
    torch.cuda.set_device(device)
    seed_offset = int(os.environ['RANK'])
else:
    device = get_device()
    local_rank = 0
    seed_offset = 0

In [ ]:
is_main = (not ddp) or (local_rank == 0)
torch.manual_seed(42 + seed_offset)

In [ ]:
# --- wandb 초기화(rank 0에서만) ---
if is_main:
    wandb.init(project='webbot-pretrain', config={})  # config는 나중에 업데이트

In [ ]:
def get_lr(it, max_lr, warmup_iters, max_iters):
    # 워밍업: 0 -> max_lr
    if it < warmup_iters:
        return max_lr * (it / warmup_iters)

    # 선형 감쇠: max_lr -> 0
    if it < max_iters:
        progress = (it - warmup_iters) / (max_iters - warmup_iters)
        return max_lr * (1.0 - progress)

    return 0.0

In [ ]:
def get_batch(data, context_len, batch_size, device, random=True, offset=0):
    if random:
        ix = torch.randint(len(data) - context_len - 1, (batch_size,))
    else:
        ix = torch.arange(offset, offset + batch_size * context_len, context_len)
        ix = ix[ix + context_len + 1 < len(data)]
        if len(ix) == 0:
            return None, None

    x = torch.stack([torch.from_numpy(data[i:i+context_len].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+context_len+1].astype(np.int64)) for i in ix])

    return x.to(device), y.to(device)

In [ ]:
def evaluate(model, val_data, context_len, batch_size, device):
    """검증: 전체 데이터를 순서대로 처리"""
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    max_start = len(val_data) - context_len - 1
    num_batches = (max_start // context_len) // batch_size + 1

    with torch.no_grad():
        for batch_idx in tqdm(range(num_batches), desc="Evaluating", leave=False):
            offset = batch_idx * batch_size * context_len

            x, y = get_batch(val_data, context_len, batch_size, device,
                        random=False, offset=offset)

            if x is None:
                break

            with autocast(device_type='cuda', dtype=torch.bfloat16):
                logits = model(x)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                    y.view(-1), reduction='sum')

            total_loss += loss.item()
            total_tokens += x.numel()

    model.train()
    return total_loss / total_tokens

In [ ]:
# --- 하이퍼파라미터 ---
vocab_size = 50000
context_len = 1024
embed_dim = 768
n_head = 12
n_kv_head = 4
n_layer = 12
ff_dim = 2048
theta = 10000

In [ ]:
micro_batch_size = 32
accumulation_steps = 4
learning_rate = 6e-4
warmup_iters = 500
max_iters = 100000
grad_clip = 1.0
eval_interval = 1000

In [ ]:
# --- wandb config 업데이트 ---
if is_main:
    wandb.config.update({
        'vocab_size': vocab_size, 'context_len': context_len,
        'embed_dim': embed_dim, 'n_head': n_head, 'n_kv_head': n_kv_head,
        'n_layer': n_layer, 'ff_dim': ff_dim, 'theta': theta,
        'micro_batch_size': micro_batch_size,
        'accumulation_steps': accumulation_steps,
        'effective_batch_size': micro_batch_size * accumulation_steps,
        'learning_rate': learning_rate, 'warmup_iters': warmup_iters,
        'max_iters': max_iters, 'grad_clip': grad_clip,
    })

In [ ]:
# --- 데이터 ---
train_data_path = 'webbot/owt_train.bin'
val_data_path = 'webbot/owt_valid.bin'
model_save_path = 'webbot/model_pretrain.pt'

In [ ]:
train_data = np.memmap(train_data_path, dtype=np.uint16, mode='r')
val_data = np.memmap(val_data_path, dtype=np.uint16, mode='r')

In [ ]:
if is_main:
    print(f"학습 데이터: {len(train_data):,} tokens")
    print(f"검증 데이터: {len(val_data):,} tokens")

In [ ]:
# --- 모델 ---
model = GPT(
    vocab_size, context_len, embed_dim, n_head,
    n_kv_head, n_layer, ff_dim, theta
).to(device)

In [ ]:
if is_main:
    num_params = sum(p.numel() for p in model.parameters())
    print(f"파라미터 수: {num_params:,} ({num_params/1e6:.1f}M)")

In [ ]:
model = torch.compile(model)

In [ ]:
if ddp:
    model = DDP(model, device_ids=[local_rank])

In [ ]:
# --- 옵티마이저 ---
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:
# --- 학습 루프 ---
pbar = tqdm(range(max_iters), disable=not is_main)
val_loss = float('inf')
val_losses = []
val_iters = []

In [ ]:
for step in pbar:
    # 학습률 갱신
    lr = get_lr(step, learning_rate, warmup_iters, max_iters)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    optimizer.zero_grad()

    # 기울기 누적 반복
    for micro_step in range(accumulation_steps):
        batch_x, batch_y = get_batch(train_data, context_len,
                                     micro_batch_size, device)

        with autocast(device_type='cuda', dtype=torch.bfloat16):
            logits = model(batch_x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   batch_y.view(-1))
            loss = loss / accumulation_steps

        # DDP: 마지막 micro_step 이외에는 기울기 동기화 비활성화
        if ddp and micro_step < accumulation_steps - 1:
            with model.no_sync():
                loss.backward()
        else:
            loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    # wandb: 매 스텝 로그 기록
    train_loss = loss.item() * accumulation_steps
    if is_main:
        wandb.log({'train/loss': train_loss, 'train/lr': lr}, step=step)

    # 주기적으로 평가
    if is_main and ((step % eval_interval) == 0 or step == max_iters - 1):
        raw_model = model.module if ddp else model
        val_loss = evaluate(raw_model, val_data, context_len,
                           micro_batch_size, device)
        val_losses.append(val_loss)
        val_iters.append(step)
        print(f"\n스텝 {step}: 검증 손실 = {val_loss:.4f}")

        wandb.log({'val/loss': val_loss}, step=step)

    if is_main:
        pbar.set_postfix({'loss': f'{train_loss:.4f}',
                          'val_loss': f'{val_loss:.4f}', 'lr': f'{lr:.2e}'})

In [ ]:
# --- 학습 곡선 저장 ---
if is_main:
    plt.figure(figsize=(10, 6))
    plt.plot(val_iters, val_losses)
    plt.xlabel('Iteration')
    plt.ylabel('Validation Loss')
    plt.grid(True)
    plt.savefig('webbot_val_loss.png')

    raw_model = model.module if ddp else model
    raw_model.save(model_save_path)
    print(f"\n모델 저장: {model_save_path}")

In [ ]:
if is_main:
    wandb.finish()

In [ ]:
if ddp:
    dist.destroy_process_group()